# Minimizar y anonimizar: tratar datos personales con cabeza

**Facsímil 9 · Seguridad, privacidad y gobernanza** — capítulo 2 (privacidad y datos personales:
minimización, DPIA y memoria).

Mandar datos personales en crudo a un servicio externo (por ejemplo, a un LLM) es de los errores más
caros —y más multables— que existen. En este cuaderno coges un montón de mensajes de soporte llenos de
datos personales y los preparas para usarlos **sin filtrar a nadie**: primero **minimizas** (lo que no
mandas no se puede filtrar), luego **detectas** lo personal, lo **seudonimizas** de forma coherente, y
compruebas con un test de **k-anonimato** —y de **l-diversidad**— si tu tabla deja a alguien señalado.
Verás también que quitar el nombre **no anonimiza** (se puede reidentificar cruzando fuentes) y cómo el
**ruido** de la privacidad diferencial protege un recuento sin delatar a nadie.

### Qué vas a aprender
- El principio de **minimización**: la primera y mejor defensa es no tener (ni mover) datos que no
  necesitas. Lo medimos en bytes.
- A **detectar** datos personales (correos, teléfonos, tarjetas, DNI, NIE, IBAN) con un único barrido de
  expresiones regulares, y por qué ningún detector es perfecto (falsos positivos y negativos).
- A **seudonimizar** de forma coherente, y en qué se diferencia de **enmascarar** y de **hashear**.
- Por qué anonimizar el texto **no basta**: **reidentificación** cruzando fuentes, **k-anonimato**,
  **l-diversidad** y **generalización**, y cómo el ruido (**privacidad diferencial**) protege agregados.

### Cuánto cuesta
Unos 18 minutos. CPU, sin claves.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
import re, json
mensajes = [
    "Hola, soy Marta Ruiz, mi telefono es 612345678 y mi correo marta.ruiz@gmail.com",
    "No me funciona la tarjeta 4111 1111 1111 1111, mi DNI es 12345678Z",
    "Llamadme al 699-112-233, pedido de Juan Perez (juan_perez@empresa.es)",
    "Mi IBAN es ES91 2100 0418 4502 0005 1332 y el cargo no aparece",
    "Soy Ana, NIE X1234567L, vivo en el CP 28013 y tengo 34 anos",
    "Telefono de contacto: seis siete ocho, nueve cero uno, dos tres cuatro",
]
print("Mensajes en crudo (NO se pueden mandar asi a ningun servicio externo):")
for m in mensajes: print("  -", m)


## 1. Minimizar: la defensa que no cuesta nada

Antes de cualquier técnica, la pregunta clave es: **¿necesito de verdad estos datos para la tarea?** Si
quieres clasificar el *tema* de un ticket, no necesitas el nombre ni el teléfono del cliente. La
minimización —no recoger, no mover, no almacenar lo que no hace falta— elimina el riesgo de raíz: lo que
no tienes, no se puede filtrar. Vamos a verlo con números: si la tarea solo necesita el **tema**, el
mensaje que viaja al servicio externo se reduce a una etiqueta diminuta.


In [ ]:
TEMAS = {
    "pago":   ["tarjeta", "cargo", "cobr", "iban", "factura"],
    "acceso": ["entrar", "contrasena", "acceso", "sesion"],
    "envio":  ["pedido", "envio", "entrega"],
}
def tema(texto):
    t = texto.lower()
    for nombre, claves in TEMAS.items():
        if any(c in t for c in claves): return nombre
    return "otro"

def minimiza(texto):
    return {"tema": tema(texto)}   # lo unico que la tarea de clasificar necesita

for m in mensajes[:4]:
    antes = len(m); despues = len(json.dumps(minimiza(m), ensure_ascii=False))
    print(f"  crudo={antes:>3} B -> minimizado={despues:>2} B   {minimiza(m)}")
print("\nLo que NO mandas no se puede filtrar: la defensa mas barata es no mover el dato.")


## 2. Detectar lo personal

Para lo que **sí** necesitas tratar, primero hay que ver dónde está lo sensible. Definimos patrones para
lo más habitual: correos, IBAN, tarjetas, DNI, NIE y teléfonos. Los combinamos en **un solo barrido** con
una expresión regular alternada: así cada trozo de texto se asigna a **un único** tipo (el primero que
encaja), y los patrones más específicos —IBAN, correo— ganan a los más generales —tarjeta, teléfono—
cuando podrían solaparse.


In [ ]:
PATRONES = {
    "EMAIL":    r"[\w.\-]+@[\w\-]+\.[\w.\-]+",
    "IBAN":     r"ES\d{2}(?:[ ]?\d{4}){5}",
    "TARJETA":  r"\b(?:\d[ -]?){13,16}\b",
    "DNI":      r"\b\d{8}[A-Za-z]\b",
    "NIE":      r"\b[XYZ]\d{7}[A-Za-z]\b",
    "TELEFONO": r"\b\d{3}[ -]?\d{3}[ -]?\d{3}\b",
}
# Un unico barrido, sin solapes: el primer patron que encaja en cada posicion se queda el trozo.
RX = re.compile("|".join(f"(?P<{t}>{p})" for t, p in PATRONES.items()))
def detecta(texto):
    return [(m.lastgroup, m.group().strip()) for m in RX.finditer(texto)]

for m in mensajes:
    encontrados = detecta(m)
    print(m[:44], "...")
    for tipo, valor in encontrados: print(f"     [{tipo}] {valor}")
    if not encontrados: print("     (nada detectado por reglas)")


In [ ]:
from collections import Counter
tipos = Counter(tipo for m in mensajes for tipo, _ in detecta(m))
print("Datos personales detectados en el lote, por tipo:")
for tipo, n in tipos.most_common(): print(f"  {tipo:<9}: {n}")
print(f"\nTotal: {sum(tipos.values())} fragmentos a proteger en {len(mensajes)} mensajes.")
print("Ojo: los NOMBRES (Marta Ruiz, Juan Perez) NO aparecen: las reglas no los pillan.")


## 3. Ningún detector por reglas es perfecto

Las expresiones regulares atrapan la mayor parte, pero fallan de dos formas: **falsos negativos** (un dato
real escrito de forma rara que se escapa) y **falsos positivos** (algo que no es personal y se marca igual).
Por eso, en serio, las reglas se combinan con **modelos** de reconocimiento de entidades, y se asume que
algo se escapará.


In [ ]:
casos = [
    ("Telefono: seis siete ocho nueve cero uno",   "telefono con letras -> NO se detecta (falso NEGATIVO)"),
    ("El pedido 612345678 es un numero interno",   "id interno de 9 cifras -> se marca TELEFONO (falso POSITIVO)"),
    ("Referencia 4929 0000 0000 0001 del catalogo","codigo de 16 cifras -> se marca TARJETA (falso POSITIVO)"),
]
for texto, nota in casos:
    print(f"  {texto}")
    print(f"     deteccion -> {detecta(texto) or 'nada'}")
    print(f"     {nota}\n")
print("Conclusion: ningun filtro de reglas es perfecto. Se combinan con modelos y se asume fuga.")


## 4. Seudonimizar de forma coherente

No basta con tachar: a veces necesitas saber que dos mensajes son de la **misma** persona sin saber
quién es. Sustituimos cada dato por una etiqueta **estable** (el mismo correo siempre da el mismo
`EMAIL_1`). Es reversible solo con la tabla de equivalencias, que guardas **aparte y cifrada**. Esto es
seudonimización (reversible con la clave), distinta de la anonimización (irreversible). Reutilizamos el
mismo barrido del detector, así que el reemplazo es coherente y sin solapes.


In [ ]:
mapa = {}; contador = {}
def seudonimo(tipo, valor):
    if valor not in mapa:
        contador[tipo] = contador.get(tipo, 0) + 1
        mapa[valor] = f"{tipo}_{contador[tipo]}"
    return mapa[valor]
def anonimiza(texto):
    return RX.sub(lambda m: seudonimo(m.lastgroup, m.group().strip()), texto)

print("Mensajes sin identificadores directos en claro (correos, telefonos, tarjetas):\n")
for m in mensajes: print("  ", anonimiza(m))
print(f"\nTabla de equivalencias (se guarda APARTE, cifrada): {len(mapa)} entradas")
print("Los nombres siguen ahi: para ellos hace falta un detector de entidades aparte.")


## 5. Seudonimizar, enmascarar, hashear: no es lo mismo

Hay tres formas de tapar un dato y se confunden a menudo:

- **Enmascarar** (redacción): tapas y punto. No se recupera **ni se enlaza**: pierdes la coherencia.
- **Seudonimizar** con tabla: estable y **reversible** solo con la tabla aparte. Permite agrupar.
- **Hashear** (resumen estable): el mismo dato da el mismo código, **enlazable** pero **no reversible**
  sin fuerza bruta. Útil cuando no necesitas volver atrás pero sí cruzar registros.

Elige según lo que la tarea necesite: cuanta menos reversibilidad, menos riesgo si se filtra la salida.


In [ ]:
import hashlib
valor = "marta.ruiz@gmail.com"
def enmascara(v): return v[0] + "***" + v[-9:]              # tapa, no enlaza
def hashea(v):    return "EMAIL_" + hashlib.sha256(v.encode()).hexdigest()[:8]
print(f"original:     {valor}")
print(f"enmascarado:  {enmascara(valor)}   (no se revierte NI se enlaza; pierdes coherencia)")
print(f"seudonimo:    {seudonimo('EMAIL', valor)}   (estable; reversible solo con la tabla aparte)")
print(f"hash estable: {hashea(valor)}   (estable y enlazable, NO reversible sin fuerza bruta)")
print("\nMismo dato -> mismo codigo en seudonimo y hash: eso es la coherencia que permite agrupar")
print("sin saber quien es. Enmascarar borra esa capacidad (y tambien el riesgo de reidentificar).")


## 6. Quitar el nombre NO anonimiza: reidentificación cruzando fuentes

El error clásico: «le he borrado el nombre, ya es anónimo». Falso. La **combinación** de atributos
aparentemente inocuos (código postal + edad + género) suele ser **única** y permite volver a poner el
nombre cruzando con otra fuente pública (un padrón, un censo, una filtración previa). Lo demostramos: dos
tablas «anónimas» que, unidas, reidentifican a la persona.


In [ ]:
publicado = [{"cp": "28001", "edad": 34, "genero": "F", "compra": "libro de cocina"}]  # sin nombre
padron    = [{"cp": "28001", "edad": 34, "genero": "F", "nombre": "Marta Ruiz"},
             {"cp": "08025", "edad": 51, "genero": "M", "nombre": "Luis Soto"}]            # fuente externa
for p in publicado:
    clave = (p["cp"], p["edad"], p["genero"])
    match = [q for q in padron if (q["cp"], q["edad"], q["genero"]) == clave]
    if match:
        print(f"Registro 'anonimo' {clave} (compra: {p['compra']})")
        print(f"   -> reidentificado como: {match[0]['nombre']}")
print("\nQuitar el nombre no basta: la COMBINACION de cuasi-identificadores reidentifica.")


## 7. ¿Tu tabla deja a alguien señalado? k-anonimato

El **k-anonimato** pone número a ese riesgo: exige que cada combinación de cuasi-identificadores (código
postal + edad + género) se repita en al menos *k* personas. Si alguien es **único** por su combinación,
está señalado: aunque le hayas borrado el nombre, se reidentifica como acabamos de ver. La k del conjunto
es el tamaño del grupo más pequeño.


In [ ]:
tabla = [
    {"cp": "28001", "edad": 34, "genero": "F"},
    {"cp": "28001", "edad": 34, "genero": "F"},
    {"cp": "28001", "edad": 34, "genero": "F"},
    {"cp": "08025", "edad": 51, "genero": "M"},   # unico en su combinacion
    {"cp": "28001", "edad": 34, "genero": "F"},
]
combos = Counter((r["cp"], r["edad"], r["genero"]) for r in tabla)
k = min(combos.values())
print("Tamano de cada grupo de cuasi-identificadores:", dict(combos))
print(f"k del conjunto = {k}", "-> ALGUIEN es unico y reidentificable!" if k < 2 else "-> ok")


## 8. La defensa: generalizar para subir la k

Para proteger a los casos únicos, **generalizamos**: en vez del código postal completo, sus 3 primeras
cifras; en vez de la edad exacta, un tramo de 10 años. Así varias personas comparten la misma
combinación y la k sube. El precio es **perder detalle**: proteger cuesta resolución, y es una decisión
consciente. Lo medimos: la k antes y después, y cuántas combinaciones distintas quedan (cuanto menos
detalle, menos combinaciones).


In [ ]:
def generaliza(r):
    return (r["cp"][:3], (r["edad"]//10)*10, r["genero"])   # CP a 3 cifras, edad por decenas

tabla2 = [
    {"cp":"28001","edad":34,"genero":"F"}, {"cp":"28004","edad":37,"genero":"F"},
    {"cp":"28010","edad":31,"genero":"F"}, {"cp":"08025","edad":51,"genero":"M"},
    {"cp":"08021","edad":55,"genero":"M"}, {"cp":"08029","edad":58,"genero":"M"},
]
antes  = Counter((r["cp"], r["edad"], r["genero"]) for r in tabla2)
despues = Counter(generaliza(r) for r in tabla2)
print(f"k SIN generalizar:  {min(antes.values())}  | {len(antes)} combinaciones distintas (cada uno unico)")
print(f"k generalizando:    {min(despues.values())}  | {len(despues)} combinaciones (varios comparten)")
print("\nGeneralizar sube la k y protege a los unicos, a costa de perder detalle: trade-off consciente.")


## 9. k no basta: l-diversidad

Puedes cumplir k-anonimato y aun así filtrar lo importante. Si un grupo de *k* personas comparte **el
mismo valor sensible** (la misma enfermedad, la misma deuda), no hace falta saber **quién** es quién: ya
sabes que **todas** lo tienen. La **l-diversidad** exige que dentro de cada grupo haya al menos *l*
valores sensibles distintos. Aquí la k está bien, pero la l es 1: protección falsa.


In [ ]:
registros = [
    {"cp": "280", "tramo": 30, "diagnostico": "depresion"},
    {"cp": "280", "tramo": 30, "diagnostico": "depresion"},
    {"cp": "280", "tramo": 30, "diagnostico": "depresion"},
]
grupo = (registros[0]["cp"], registros[0]["tramo"])
valores = {r["diagnostico"] for r in registros}
print(f"Grupo {grupo}: k = {len(registros)}  (parece protegido)")
print(f"Valores sensibles distintos en el grupo: l = {len(valores)} -> {valores}")
print("Aunque no sepas QUIEN es cada uno, sabes que TODOS tienen el mismo diagnostico.")
print("La k no protege el atributo sensible; eso lo mide la l-diversidad (aqui l=1: malo).")


## 10. Proteger un recuento: privacidad diferencial

A veces no publicas filas, sino **agregados** («cuántos clientes tienen tal condición»). Incluso ahí, si
alguien sabe el resto, puede deducir si **una** persona concreta está dentro. La **privacidad diferencial**
añade **ruido** calibrado al resultado: oculta el aporte de cualquier individuo, pero el agregado sigue
siendo útil porque el ruido se promedia a cero sobre muchas consultas.


In [ ]:
import numpy as np
np.random.seed(0)
verdad = 42   # nº real de clientes con cierta condicion
def consulta_privada(valor, epsilon=1.0):
    return valor + np.random.laplace(0, 1/epsilon)   # mas epsilon = menos ruido = menos privacidad

muestras = [consulta_privada(verdad) for _ in range(5)]
media_larga = np.mean([consulta_privada(verdad) for _ in range(2000)])
print(f"Conteo real:                 {verdad}")
print("Respuestas con ruido (5):   ", [f"{x:.1f}" for x in muestras])
print(f"Media de 2000 consultas:     {media_larga:.1f}  (el ruido se promedia a cero)")
print("\nEl ruido oculta si UNA persona esta o no en el conteo; el agregado sigue siendo util.")


## 11. Pruébalo tú

1. **Añade un patrón de matrícula** (`r"\b\d{4}[ -]?[A-Z]{3}\b"`) y un mensaje que la contenga. ¿Qué otros
   datos personales se te ocurren (número de la Seguridad Social, IP)?
2. **Rompe el detector:** escribe un teléfono como «seis uno dos...». Los regex no lo pillan; por eso en
   serio se combinan con modelos. Ningún filtro es perfecto.
3. **Sube el nivel de generalización** (edad por tramos de 20, CP a 2 cifras): ¿llegas a k≥3 para todos?
   ¿Cuántas combinaciones distintas pierdes?
4. **Baja epsilon** en la privacidad diferencial (más ruido): ¿a partir de qué punto el recuento deja de
   ser útil? Protección y utilidad tiran en sentidos opuestos.
5. **l-diversidad:** cambia un diagnóstico del grupo y observa cómo sube la l. ¿Cuánta diversidad crees
   que hace falta para estar tranquilo?


## 12. Errores comunes

- **Saltarse la minimización.** La defensa más barata es no tener el dato. Pregúntate siempre si lo
  necesitas antes de recogerlo o moverlo.
- **Confiar el 100% en los regex.** Se escapan formatos raros (falsos negativos) y marcan lo que no es
  (falsos positivos). Combínalos con modelos y asume que algo pasará.
- **Creer que borrar el nombre anonimiza.** La combinación de cuasi-identificadores reidentifica cruzando
  fuentes. Por eso se mide la k.
- **Quedarse en el k-anonimato.** Si todo el grupo comparte el valor sensible, la l-diversidad falla y
  filtras igual.
- **Confundir seudonimizar, enmascarar y hashear.** Cambian en reversibilidad y en si permiten enlazar;
  no son lo mismo a efectos legales ni de riesgo.


## 13. Qué te llevas

- **Minimizar** (no mover lo que no hace falta) es la primera defensa y se mide: el ticket que viaja puede
  reducirse a una etiqueta diminuta.
- Ningún detector es perfecto: se combinan reglas y modelos y se asume fuga. **Seudonimizar**, **enmascarar**
  y **hashear** no son lo mismo: difieren en reversibilidad y en si dejan enlazar.
- Anonimizar el texto no basta: quitar el nombre **no anonimiza** (reidentificación cruzando fuentes); el
  **k-anonimato** vigila la combinación, la **l-diversidad** el atributo sensible, y la **privacidad
  diferencial** protege agregados con ruido. Proteger siempre cuesta detalle: es un trade-off consciente.

**Para seguir:** el capítulo 3 trata la seguridad de las aplicaciones LLM (el siguiente cuaderno: inyección
de prompt); el capítulo 4, el cumplimiento y los paquetes de evidencia.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*